In [3]:
%%capture
import os
os.environ["HTTP_PROXY"]="http://proxy.csn29.bessy.de:3128"
os.environ["HTTPS_PROXY"]="http://proxy.csn29.bessy.de:3128"
!pip install ipywidgets

!pip uninstall insitu_analyser -y
!pip cache purge
!pip install git+https://codebase.helmholtz.cloud/hzb-se-alm/insitu_analyser.git@develop
# !pip install git+https://codebase.helmholtz.cloud/lennart.reb/insight_hzb@dev

In [4]:
%%capture
import matplotlib.pyplot as plt
from insitu_analyser.Preview.perfect_previewer import PERFECTPREVIEWER


In [5]:
plt.ioff()
%matplotlib widget

import os
url_base = "http://nomad03.csn29.bessy.de"
url = f"{url_base}/nomad-oasis/api/v1"
token = os.environ['NOMAD_CLIENT_ACCESS_TOKEN']

In [6]:
import ipywidgets as widgets
from insitu_analyser.utils.nomad_api_calls import get_specific_data_of_sample, get_samples_in_upload, get_sample_description
from insitu_analyser.utils.search_bar_widget import get_searchbar_widgets, get_upload_folder, create_spinner
import base64, gc
from IPython.display import clear_output, display, HTML

display(HTML("""
<style>
/* Only apply horizontal scroll to actual output area */
.output_subarea {
    overflow-x: auto !important;
    width: 100% !important;
}

/* Prevent nested divs from adding more scrollbars */
.jp-OutputArea-output > div, 
.output_subarea > div, 
.widget-output > div {
    overflow-x: visible !important;
}
</style>
"""))

spinner = create_spinner()

P = None
analysis_widgets = None
optical_2 = None
logging = None
cuts = None
comparison = None
csv_download = None
thickness = None
peak_analyzer = None

def free_storage():
    global P, analysis_widgets, optical_2, logging, cuts, comparison_1, comparison_2,csv_download, thickness, peak_analyzer
    try:
        plt.close('all')
        clear_output()
        del P.G.image
        del P.G
        del P.H
        del P
        del analysis_widgets
        del optical_2
        del logging
        del cuts
        del comparison_1
        del comparison_2
        del csv_download
        del thickness
        del peak_analyzer
        gc.collect()
    except:
        pass

def on_select_sample(b):
    global P, analysis_widgets, optical_2, logging , cuts, comparison_1, comparison_2, csv_download, thickness, peak_analyzer, file_path
    out.clear_output()
    measurements = get_specific_data_of_sample(url, token, samples.value.split(" [")[0], "HySprint_Process", with_meta=True)
    options = []
    for m in measurements:
        for file in m[0]["data_file"]:
            if not file.endswith("h5"):
                continue
            if "description" in m[0]:
                described = m[0]["description"]
            else:
                described = ""
            options.append((described + "---" + file, os.path.join("..",get_upload_folder(m[1]["upload_id"]), file)))
            
    file_path.options = options
    

def on_select_file(b):
    global P, analysis_widgets, optical_2, logging , cuts, comparison_1, comparison_2, csv_download, thickness, peak_analyzer, file_path
    out.clear_output()
    with out:
        display(spinner)
    free_storage()
    if file_path.value:
        P = PERFECTPREVIEWER(file_path.value, slider_size="70%", screenwidth=pixelwidth.value, nomad_url=url_base + "/")
        analysis_widgets = P.display_widgets(xrd=True,optical=True)
        optical_2 = P.display_optical_data()
        logging = P.display_logging()
        cuts = P.display_cuts()
        comparison_1, comparison_2 = P.display_comparison()
        csv_download = P.display_export()
        thickness = P.link_thickness()
        peak_analyzer = P.link_peak_analyzer()
        out.clear_output()
        with out:
            display(analysis_widgets["giwaxs_content"])
            display(analysis_widgets["ui"])
            display(analysis_widgets["optical_content"])
            display(optical_2)
            display(cuts)
            display(logging)
            display(comparison_1)
            display(comparison_2)
            display(csv_download)
            display(thickness)
            display(peak_analyzer)
        gc.collect()         

out = widgets.Output()
search_field, search, batch_ids, samples, file_path, pixelwidth = get_searchbar_widgets(url, token)
for w in [samples, pixelwidth, file_path]:
    w.observe(on_select_sample, names=['value'])

file_path.observe(on_select_file, names=['value'])

upload_id = os.getcwd()[-22:]
sample_ids = get_samples_in_upload(url,token,upload_id)
sample_description = get_sample_description(url, token, sample_ids)
samples.options = sample_description

search_box = widgets.VBox([search_field, search])
h_box = widgets.HBox([search_box, batch_ids, samples,file_path])
display(widgets.VBox([h_box, pixelwidth]))


In [7]:
display(out)

Output()